In [1]:
from tqdm.notebook import tqdm
import matplotlib.pyplot as plt
import numpy as np

import torch
import torch.nn as nn
from torch.nn import Linear, ReLU, MSELoss, L1Loss, Sequential, Conv2d, ConvTranspose2d, MaxPool2d, AdaptiveAvgPool2d, Module, BatchNorm2d, Sigmoid, Dropout
import torch.optim as optim
from torch.autograd import Variable
from torch.utils.data import random_split
import os
from PIL import Image
import cv2
import torchvision.transforms as transforms
import torch.nn.functional as F
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader
import pywt
import numpy as np
from PIL import Image

/home/netweb/anaconda3/envs/ggma/lib/python3.6/site-packages/tqdm/auto.py:22: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Using device:", device)

Using device: cuda


In [3]:
class autoencoders(nn.Module):
    def __init__(self):
        super(autoencoders, self).__init__()
        self.encoder = torch.nn.Sequential(
            Conv2d(3, 32, kernel_size = (3,3), padding = "same"),
            ReLU(),
            nn.BatchNorm2d(32),
            MaxPool2d((2,2), padding = 0),
            Conv2d(32, 64, kernel_size = (3,3), padding = "same"),
            ReLU(),
            nn.BatchNorm2d(64),
            MaxPool2d((2,2), padding = 0),
            Conv2d(64, 128, kernel_size = (3,3), padding = "same"),
            ReLU(),
            MaxPool2d((2,2), padding = 0)
        )
        self.decoder = torch.nn.Sequential(
            ConvTranspose2d(128, 128, kernel_size = (3,3), stride = 2, padding = 0),
            ReLU(),
            nn.BatchNorm2d(128),
            ConvTranspose2d(128, 64, kernel_size = (3,3), stride = 2, padding = 2),
            ReLU(),
            nn.BatchNorm2d(64),
            ConvTranspose2d(64, 32, kernel_size = (3,3), stride = 1, padding = 0),
            ReLU(),
            ConvTranspose2d(32, 3, kernel_size = (2,2), stride = 2, padding = 1),
            Sigmoid()
        )
        
    def forward(self, images):
        x = self.encoder(images)
        x = self.decoder(x)
        return x

In [4]:
autoencoder=autoencoders().to(device)

optimizer = optim.Adam(autoencoder.parameters(), lr=3e-3)

model_name=os.path.join("dae.pth")
state = torch.load(model_name)
autoencoder.load_state_dict(state['DAE_weights'])
optimizer.load_state_dict(state['opt'])

In [5]:
transform = transforms.Compose([
    transforms.Resize((128,128)),
    transforms.ToTensor()
])

dataset = ImageFolder("train", transform=transform)

loader = DataLoader(
    dataset,
    batch_size=4,
    shuffle=True,
    num_workers=4
)

In [6]:
class DWT2D(nn.Module):

    def __init__(self):
        super().__init__()

    def forward(self, x):
        B, C, H, W = x.shape
        outputs = []

        for i in range(B):

            img = x[i].permute(1,2,0).cpu().numpy()
            channels = []

            for c in range(C):

                coeffs2 = pywt.dwt2(img[:,:,c], 'haar')
                LL,(LH,HL,HH) = coeffs2

                channels.append(LH)
                channels.append(HL)
                channels.append(HH)

            channels = np.stack(channels)
            channels = torch.tensor(channels)

            outputs.append(channels)

        outputs = torch.stack(outputs).float().to(x.device)

        return outputs

In [7]:
import torch.nn.functional as F

class SpatialMultiHeadAttention(nn.Module):

    def __init__(self, in_channels, heads=4):
        super().__init__()

        self.attn = nn.MultiheadAttention(
            embed_dim=in_channels,
            num_heads=heads,
            batch_first=True
        )

    def forward(self, x):

        B,C,H,W = x.shape

        x_small = F.avg_pool2d(x, 4)   

        Bs,Cs,Hs,Ws = x_small.shape

        x_small = x_small.view(Bs,Cs,Hs*Ws).permute(0,2,1)

        attn_out,_ = self.attn(x_small,x_small,x_small)

        attn_out = attn_out.permute(0,2,1).view(Bs,Cs,Hs,Ws)

        attn_out = F.interpolate(attn_out, size=(H,W), mode="bilinear")

        return attn_out

In [8]:
class AbstractorNetwork(nn.Module):

    def __init__(self, autoencoder):

        super().__init__()

        self.autoencoder = autoencoder
        self.dwt = DWT2D()

        self.attention = SpatialMultiHeadAttention(
            in_channels=12,   
            heads=4
        )

    def forward(self, x):

        with torch.no_grad():
            recon = self.autoencoder(x)

        R = torch.abs(x - recon)

        W = self.dwt(x)

        W = F.interpolate(
            W,
            size=(x.shape[2], x.shape[3]),
            mode="bilinear"
        )

        J = torch.cat([R, W], dim=1)

        Z = self.attention(J)

        return Z

In [9]:
import torch
import torch.nn as nn
import torchvision.models as models

class FeatureExtractor(nn.Module):

    def __init__(self):
        super().__init__()

        backbone = models.resnet50(pretrained=True)

        backbone.conv1 = nn.Conv2d(
            12,
            64,
            kernel_size=7,
            stride=2,
            padding=3,
            bias=False
        )

        self.encoder = nn.Sequential(*list(backbone.children())[:-1])

    def forward(self,x):

        x = self.encoder(x)

        x = torch.flatten(x,1)

        return x

In [10]:
class ProjectionHead(nn.Module):

    def __init__(self):

        super().__init__()

        self.mlp = nn.Sequential(
            nn.Linear(2048,2048),
            nn.ReLU(),
            nn.Linear(2048,128)
        )

    def forward(self,x):
        return self.mlp(x)

In [11]:
class ClassificationHead(nn.Module):

    def __init__(self,num_classes=12):

        super().__init__()

        self.fc = nn.Sequential(
            nn.Linear(2048,1024),
            nn.ReLU(),
            nn.Linear(1024,512),
            nn.ReLU(),
            nn.Linear(512,256),
            nn.ReLU(),
            nn.Linear(256,num_classes)
        )

    def forward(self,x):
        return self.fc(x)

In [13]:
class GGMA_Net(nn.Module):

    def __init__(self,abstractor,num_classes=12):

        super().__init__()

        self.abstractor = abstractor

        self.feature_extractor = FeatureExtractor()

        self.projection_head = ProjectionHead()

        self.classifier = ClassificationHead(num_classes)

    def forward(self,x):

        z = self.abstractor(x)

        embedding = self.feature_extractor(z)

        proj = self.projection_head(embedding)

        logits = self.classifier(embedding)

        return embedding,proj,logits

In [20]:
checkpoint = torch.load("gta_model.pth", map_location=device)
abstractor = AbstractorNetwork(autoencoder)
model = GGMA_Net(abstractor, num_classes=12)
model.load_state_dict(checkpoint["model_state_dict"])

class_names = checkpoint["class_names"]

model = model.to(device)
model.eval()

print("GTA model loaded")



def preprocess_image(path):
    img = Image.open(path).convert("RGB")
    img = transform(img)
    return img


def predict(image_input):


    if isinstance(image_input, str):

        batch = torch.stack([preprocess_image(image_input)])

    elif isinstance(image_input, list):

        batch = torch.stack([preprocess_image(p) for p in image_input])

    elif torch.is_tensor(image_input):

        batch = image_input

    else:
        raise ValueError("Input must be image path, list of paths, or tensor")

    batch = batch.to(device)

    with torch.no_grad():

        _, _, logits = model(batch)

        probs = torch.softmax(logits, dim=1)

        preds = torch.argmax(probs, dim=1)

    results = []

    for i in range(len(preds)):

        results.append({
            "class": class_names[preds[i].item()]
        })

    return results






GGMA model loaded


In [22]:

print(predict("img000018.jpg"))



[{'class': 'ldm'}]
